# ESO Adaptive Optics Telemetry Benchmark
This notebook dynamically compiles the Project Radius C Engine to benchmark real-world ESO telemetry (AOT) files at ultra-low latency.


## 1. Environment Setup
Install `aotpy` (Adaptive Optics Telemetry standard library) and dependencies.


In [ ]:
!pip install aotpy astropy numpy scipy


## 2. Dynamic C Engine Architecture
We inject our modified C Engine source code capable of handling dynamic array sizes based on VLT telemetry metadata.


In [ ]:
%%writefile geometry.h
#ifndef GEOMETRY_H
#define GEOMETRY_H

typedef struct {
    int grid_size;
    double pitch;
    double focal_length;
    double wavelength;
    int num_subapertures; // Computed from active pixels
} LensletConfig;

#endif



In [ ]:
%%writefile mvm_reconstructor.c
#include <stdlib.h>
#include <stdio.h>
#include "geometry.h"

// Dynamic MVM Reconstruction
// G_plus: The inverted interaction matrix (size: num_zernikes x (2 * num_subapertures))
// slopes: The input slopes vector (size: 2 * num_subapertures)
// zernikes: The output vector (size: num_zernikes)
void reconstruct_zernikes(double* G_plus, double* slopes, double* zernikes, int num_zernikes, int num_slopes) {
    for (int i = 0; i < num_zernikes; i++) {
        zernikes[i] = 0.0;
        for (int j = 0; j < num_slopes; j++) {
            zernikes[i] += G_plus[i * num_slopes + j] * slopes[j];
        }
    }
}



In [ ]:
%%writefile slopes.c
#include <stdlib.h>
#include <math.h>
#include "geometry.h"

// Dummy centroiding function for benchmark purposes. 
// In full ESO integration, we extract centroid data from aotpy directly or compute over pixels.
void compute_slopes_dynamic(double* image_data, double* slopes, LensletConfig config) {
    // Fill with dummy slopes to benchmark pure execution speed
    for(int i=0; i < config.num_subapertures * 2; i++) {
        slopes[i] = 0.0001; 
    }
}



## 3. Compile C Engine
Compile the injected C code into a high-performance shared library.


In [ ]:
!gcc -shared -o c_engine.so -fPIC slopes.c mvm_reconstructor.c -O3 -march=native


## 4. Download Real ESO Telemetry (ERIS SCAO)
We download a real-world ESO AOT FITS file from the Zenodo archive.


In [ ]:
!wget -O ERIS_NGS.fits 'https://zenodo.org/records/8192742/files/ERIS_NGS_20230331_012546.fits?download=1'


## 5. Benchmark Execution with Real Data
We load the FITS file using `aotpy`, extract the raw slopes, and feed them into our C Engine MVM.


In [ ]:
import ctypes
import time
import numpy as np
import os
import aotpy

# Load the compiled library
c_engine = ctypes.CDLL(os.path.abspath('./c_engine.so'))

print("Loading ERIS NGS Telemetry...")
system = aotpy.AOSystem.read_from_file("ERIS_NGS.fits")

# Find the wavefront sensor slopes (measurements)
wfs = system.wavefront_sensors[0]
slopes_data = wfs.measurements.data # Shape is usually (time_steps, num_slopes)

num_frames = slopes_data.shape[0]
num_slopes = slopes_data.shape[1]
num_zernikes = 50 # We map to 50 Zernike modes

print(f"Loaded {num_frames} frames with {num_slopes} slopes each.")

# Setup random pseudo-inverse Matrix G+ (to simulate the precomputed matrix matching ERIS)
G_plus = np.random.randn(num_zernikes, num_slopes).astype(np.float64)

# Setup array bindings
c_reconstruct = c_engine.reconstruct_zernikes
c_reconstruct.argtypes = [
    np.ctypeslib.ndpointer(dtype=np.float64, ndim=2, flags='C_CONTIGUOUS'),
    np.ctypeslib.ndpointer(dtype=np.float64, ndim=1, flags='C_CONTIGUOUS'),
    np.ctypeslib.ndpointer(dtype=np.float64, ndim=1, flags='C_CONTIGUOUS'),
    ctypes.c_int,
    ctypes.c_int
]

zernikes_out = np.zeros(num_zernikes, dtype=np.float64)

# Benchmark Execution Speed
print("Executing C Engine MVM Reconstructor on REAL TELEMETRY...")
start_time = time.perf_counter()

for i in range(num_frames):
    frame_slopes = slopes_data[i].astype(np.float64).flatten()
    c_reconstruct(G_plus, frame_slopes, zernikes_out, num_zernikes, num_slopes)

end_time = time.perf_counter()
total_time = end_time - start_time
time_per_frame = (total_time / num_frames) * 1000  # in ms

print(f"Total time for {num_frames} real frames: {total_time:.4f} seconds")
print(f"Average Latency per frame: {time_per_frame:.4f} milliseconds")

if time_per_frame < 10.0:
    print("\nSUCCESS: The C Engine comfortably beats the 10ms real-time deadline for real ERIS telemetry!")
else:
    print("\nFAIL: Latency exceeds limits.")

